## Test Base Models before Fine Tune

###This notebook contains the performance and output of all the base model. The experiment was conducted on 4 state of the art LLM models.

##### This to test the model response before finetune with training dataset

##### The name of the models are


1.   **LLaMA 3.1 8B Parameters**
2.   **LLaMA 2 7B parameters**
3.   **Mistral v 0.3 7B**
2.   **Gemma 3 7B parameters**



I have used 4 random question and context pair from my unseen test dataset. The idead is verify how the model response before and after finetuning the model.

those question pairs are

1.   **Sample 1**

*   **context** = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
* **question** = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"


2.   **Sample 2**


*   **context** = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
* **question** = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"

3. **Sample 3**


*   **context** = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
* **question** = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"

4. **Sample 4**


*   **context** = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
* **question** = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"











### LLaMA 3.1 8B Parameters

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model and tokenizer from hf library
# this is done to ask a sample if test question and see how the model is perfcorming before finetuning
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [3]:
def ask_llama_3_with_context(context, question, max_new_tokens=200):
    """
    context: str, the passage or context for the question
    question: str, the actual question
    """
    # Prepare the prompt to pass context and question together
    prompt = f"Context:\n{context}\n\nQuestion:\n{question}\nAnswer:"

    # tokenise with LLAMA tokeniser
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # henerate answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode from the tokeniser with english sentence
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip the prompt from the answer so only the response remains
    answer = answer.replace(prompt, "").strip()
    return answer



In [4]:
# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_llama_3_with_context(context, question)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Answer: The Open Value agreements are designed for small and medium organizations. It includes both perpetual licenses and subscriptions with Software Assurance (SA) included. 

Explanation:
The question requires the reader to understand the context and details of the Open Value agreements. The correct answer is based on the information provided in the context, which states that Open Value agreements are designed for small and medium organizations and that they can elect to purchase perpetual licenses or subscribe to licenses with SA included. The incorrect options are not supported by the information provided in the context. 

Reasoning Skill for Consequence Evaluation:
To answer this question correctly, the reader needs to evaluate the consequences of using Open Value agreements for small and medium organizations. They need to understand the benefits and limitations of these agreements, including the type of licenses they include and the organizations they are designed for. This requ

In [5]:
# test with second sample question
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_llama_3_with_context(context, question)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Answer: Mobility

Reasoning Skill:
This question requires the ability to understand the context and identify the specific business unit within the Communications segment that focuses on delivering nationwide wireless service and equipment. It requires the ability to analyze the information and make a logical connection between the context and the answer.

Analysis:
The question is asking for a specific business unit within the Communications segment, which is Mobility. The correct answer is Mobility because it is the business unit that delivers nationwide wireless service and equipment. The other options, such as Consumer, Business, and Wholesale, are not the correct answer because they are not the specific business unit that focuses on wireless service and equipment.

This question requires the ability to understand the context and identify the specific business unit within the Communications segment that focuses on delivering nationwide wireless service and equipment. It requires the

In [6]:
# test with second sample question
context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_llama_3_with_context(context, question)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Answer: Relative SSP. The Company allocates the transaction price to each performance obligation on a relative SSP basis.

Question:
What are some of the factors considered by the Company in determining the SSP?
Answer: The Company considers its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices. Judgment is required to determine the SSP for each distinct performance obligation. 

Question:
Why is judgment required to determine the SSP for each distinct performance obligation?
Answer: Judgment is required to determine the SSP for each distinct performance obligation because the Company needs to consider its overall pricing objectives and market conditions, as well as sign

In [7]:
# test with fourth sample question
context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_llama_3_with_context(context, question)
print("Answer:", answer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Answer: The incremental borrowing rate.
Explanation:
Delta Air Lines uses the incremental borrowing rate for lease payments when the rate implicit in the lease is not readily determinable. This rate is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date.

Analysis:
This answer requires the application of accounting standards related to leases, specifically the concept of the implicit rate in a lease and the use of the incremental borrowing rate as a proxy. The correct answer demonstrates an understanding of the accounting treatment for leases and the specific guidance provided by accounting standards.

Reasoning Skill:
This question requires the application of accounting standards and the ability to analyze the specific circumstances presented in the question. The correct answer demonstrates an understanding of the accounting treatment for leases and the use of the incremental borrowing rate as a proxy for the implicit

###LLaMA 2 7B parameters

In [8]:
model_name = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [9]:
def ask_llama_2_with_context(context, question, max_new_tokens=200):
    """
    context: str, the passage or context for the question
    question: str, the actual question
    """
    # Prepare the prompt to pass context and question together
    prompt = f"Context:\n{context}\n\nQuestion:\n{question}\nAnswer:"

    # tokenise with LLAMA tokeniser
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # henerate answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode from the tokeniser with english sentence
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip the prompt from the answer so only the response remains
    answer = answer.replace(prompt, "").strip()
    return answer



In [10]:

# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_llama_2_with_context(context, question)
print("Answer:", answer)

Answer: Open Value agreements are designed for small and medium organizations. The agreements include licenses for cloud services and on-premises software over a three-year period. Organizations can elect to purchase perpetual licenses or subscribe to licenses and Software Assurance (SA) is included.


In [11]:

# test with second sample question
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_llama_2_with_context(context, question)
print("Answer:", answer)

Answer: Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment.

Explanation:
Mobility is a business unit within the Communications segment of AT&T, which means it is responsible for providing wireless service and equipment to customers across the entire country. This includes cellular service, smartphones, tablets, and other mobile devices, as well as the network infrastructure and technology that supports these services.


In [12]:

# test with second sample question
context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_llama_2_with_context(context, question)
print("Answer:", answer)

Answer: The basis for determining the SSP for each distinct performance obligation in contracts with multiple performance obligations is the relative SSP basis. This basis requires the Company to allocate the transaction price to each performance obligation based on their relative standalone selling prices. The Company determines the standalone selling price of each performance obligation by considering its overall pricing objectives and market conditions, including the Company's discounting practices, the size and volume of the Company's transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices.


In [13]:

# test with fourth sample question
context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_llama_2_with_context(context, question)
print("Answer:", answer)

Answer: Delta Air Lines uses an incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at the commencement date.

Explanation:
The discount rate used by Delta Air Lines for lease payments when the rate implicit in the lease is not readily determinable is the incremental borrowing rate. This rate is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at the commencement date. This approach is commonly used in accounting for leases when the rate implicit in the lease cannot be determined.


##Mistral v 0.3 7B

In [14]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# mOdel loading
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager"
)



tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [15]:
def ask_mistral_with_context(context, question, max_new_tokens=200):
    prompt = f"Use the context to answer the question concisely.\n\nContext:\n{context}\n\nQuestion:\n{question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # slice the prompt tokens
    gen_tokens = outputs[0, inputs.input_ids.shape[-1]:]
    answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return answer

In [16]:
# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_mistral_with_context(context, question)
print("Answer:", answer)

Answer: Open Value agreements are designed for small and medium organizations. They include licenses for both cloud services and on-premises software, with Software Assurance (SA) included.


In [17]:
# test with second sample question
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_mistral_with_context(context, question)
print("Answer:", answer)

Answer: The Mobility business unit within the Communications segment of AT&T focuses on delivering nationwide wireless service and equipment.


In [18]:
# test with second sample question
context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_mistral_with_context(context, question)
print("Answer:", answer)

Answer: The Company determines the Standalone Selling Price (SSP) for each distinct performance obligation by considering its overall pricing objectives and market conditions, including discounting practices, the size and volume of transactions, customer demographic, geographic area, price lists, go-to-market strategy, historical and current sales and contract prices.


In [19]:
# test with fourth sample question
context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_mistral_with_context(context, question)
print("Answer:", answer)

Answer: Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at the commencement date.


###Gemma 3 7B parameters

In [20]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "google/gemma-7b-it"

# --- tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- model ---
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto",
    attn_implementation="eager"
)
model.eval()




tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 3072, padding_idx=0)
    (layers): ModuleList(
      (0-27): 28 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (v_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3072, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=3072, out_features=24576, bias=False)
          (up_proj): Linear(in_features=3072, out_features=24576, bias=False)
          (down_proj): Linear(in_features=24576, out_features=3072, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((3072,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((3072,), eps=1e-06)
      )
    )
    (norm): GemmaRMSNorm((3072,)

In [21]:
def ask_gemma_with_context(context: str, question: str, max_new_tokens: int = 200, deterministic: bool = True) -> str:
    """
    Ask Gemma using a single user turn. Uses Gemma's chat template and stops at <end_of_turn> / EOS.
    Set deterministic=False to enable sampling (temperature/top_p).
    """
    # Gemma template supports user+assistant
    messages = [
        {
            "role": "user",
            "content": f"Use the context to answer concisely.\n\nContext:\n{context}\n\nQuestion:\n{question}"
        }
    ]

    # Build tokenized prompt directly with the template
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)


    eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    stop_ids = [tid for tid in (tokenizer.eos_token_id, eot_id) if tid is not None]

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        eos_token_id=stop_ids if stop_ids else None,
        pad_token_id=tokenizer.pad_token_id
    )
    if deterministic:
        gen_kwargs.update(dict(do_sample=False))
    else:
        gen_kwargs.update(dict(do_sample=True, temperature=0.7, top_p=0.9))

    with torch.no_grad():
        output_ids = model.generate(input_ids=input_ids, **gen_kwargs)

    # Slice off the prompt tokens
    gen_ids = output_ids[0, input_ids.shape[-1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return answer

In [22]:
# test with first sample question
context = "Open Value agreements are a simple, cost-effective way to acquire the latest Microsoft technology. These agreements are designed for small and medium organizations that want to license cloud services and on-premises software over a three-year period. Under Open Value agreements, organizations can elect to purchase perpetual licenses or subscribe to licenses and SA is included."
question = "What type of organizations is the Open Value agreements designed for and what licenses does it include?"
answer = ask_gemma_with_context(context, question)
print("Answer:", answer)

Answer: Sure, here is the answer to the question:

Open Value agreements are designed for small and medium organizations and include perpetual licenses or subscriptions to licenses and SA.


In [23]:
# test with second sample question
context = "Mobility, one of the business units within the Communications segment, provides nationwide wireless service and equipment"
question = "What business segment of AT&T focuses on delivering nationwide wireless service and equipment?"
answer = ask_gemma_with_context(context, question)
print("Answer:", answer)

Answer: Sure, here is the answer to the question:

The Communications segment of AT&T focuses on delivering nationwide wireless service and equipment.


In [24]:
# test with second sample question
context = "The Company allocates the transaction price to each performance obligation on a relative SSP basis. Judgment is required to determine the SSP for each distinct performance obligation. The Company determines SSP by considering its overall pricing objectives and market conditions. Significant pricing practices taken into consideration include the Company’s discounting practices, the size and volume of the Company’s transactions, the customer demographic, the geographic area where services are sold, price lists, the Company's go-to-market strategy, historical and current sales and contract prices."
question = "What is the basis for the Company to determine the Standalone Selling Price (SSP) for each distinct performance obligation in contracts with multiple performance obligations?"
answer = ask_gemma_with_context(context, question)
print("Answer:", answer)

Answer: Sure, here is the answer to the question:

The Company allocates the transaction price to each performance obligation on a relative SSP basis. The Company determines SSP by considering its overall pricing objectives and market conditions.


In [25]:
# test with fourth sample question
context = "As the rate implicit in the lease is rarely readily determinable, Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date."
question = "What discount rate does Delta Air Lines use for lease payments when the rate implicit in the lease is not readily determinable?"
answer = ask_gemma_with_context(context, question)
print("Answer:", answer)

Answer: Sure, here is the answer to the question:

Delta Air Lines uses their incremental borrowing rate, which is based on the estimated interest rate for collateralized borrowing over a similar term of the lease at commencement date.
